# Subword Tokenization as Compression: BPE, WordPiece, and Unigram LM

A word vocabulary is closed; the vocabulary of text is not (Topic 1: Heaps' law needs an open
vocabulary, and a closed word list saturates). Subword tokenizers close the gap by spelling rare words
from a fixed inventory of pieces. This notebook walks the topic's argument with every number computed by
`subword_tokenization_bpe_unigram.py`, which owns them; nothing here redefines the math.

The arc: a vocabulary encodes everything iff it contains the alphabet, but a word has exponentially many
segmentations (§1); BPE greedily merges the most frequent pair, which compresses but is not optimal (§2);
WordPiece merges by likelihood gain, a count-weighted PMI (§3); the Unigram LM scores whole segmentations
and is trained by EM (§4); all three are codes, and their lengths can be compared (§5); and the finance
corpus plants a known morphology to test which boundaries each tokenizer finds (§6).

In [ ]:
import math
import sys
from collections import Counter
from pathlib import Path

here = Path.cwd()
sys.path.insert(0, str(here if (here / "subword_tokenization_bpe_unigram.py").exists()
                       else here / "notebooks" / "subword-tokenization-bpe-unigram"))
import subword_tokenization_bpe_unigram as sw

fw = sw.finance_words()
print(f"training: {sum(fw['train'].values())} tokens, {len(fw['train'])} word types "
      f"({len(fw['pseudo'])} planted pseudo-words)")
print(f"held out: {sum(fw['held'].values())} tokens; {len(fw['oov'])} word types never seen in training "
      f"({sum(fw['oov'].values()) / sum(fw['held'].values()):.2%} of held-out tokens)")
print("some OOV words:", sorted(fw["oov"])[:8])

## 1. The open-vocabulary problem

Fix an alphabet $\Sigma$ and a finite vocabulary $V \subset \Sigma^+$. A **segmentation** of a string
$x$ is a tuple $(t_1, \dots, t_k) \in V^k$ with $t_1 t_2 \cdots t_k = x$; write $\mathrm{Seg}_V(x)$ for the set.

**Proposition 1 (completeness).** $\mathrm{Seg}_V(x) \neq \varnothing$ for every $x \in \Sigma^+$ iff
$\Sigma \subseteq V$. *(If $\Sigma \subseteq V$, spell $x$ character by character; if some $c \in \Sigma$ is
missing, the one-character string $c$ has no segmentation.)* Decoding is concatenation, so it inverts
every segmentation.

**Proposition 2 (counting).** If every substring of $x$ is a piece, $|\mathrm{Seg}(x)| = 2^{n-1}$ for
$|x| = n$: each of the $n-1$ gaps is a boundary or not. With pieces capped at length $L$ the count obeys
$S(m) = S(m-1) + \dots + S(m-L)$, the $L$-step Fibonacci numbers. Choosing a segmentation is therefore a
search problem, and §4 solves it by dynamic programming.

In [ ]:
sw.test_completeness_iff_alphabet_in_vocab()
sw.test_segmentation_count_is_lbonacci()
print("segmentations of a 16-letter word, all substrings allowed:", sw.count_segmentations("x" * 16))
print("pieces capped at L = 3:", sw.lbonacci(16, 3))

## 2. BPE: greedy compression

Byte-pair encoding starts from $V_0 = \Sigma$ and repeats: count every adjacent pair of tokens in the
training words, merge the most frequent pair $(a, b)$ into the new token $ab$ everywhere, and add $ab$ to
the vocabulary. The **merge sequence** $\mu = ((a_1, b_1), \dots, (a_M, b_M))$ is the model.

**Proposition 3 (merge accounting).** Merging $(a, b)$ left to right without overlaps shortens a token
sequence by exactly the number $r$ of replacements made. If $a \neq b$, two occurrences of $(a, b)$ cannot
overlap, so $r$ equals the window count $c(a, b)$; if $a = b$, a run of $\ell$ copies of $a$ holds $\ell - 1$
windows but yields only $\lfloor \ell/2 \rfloor$ replacements — the count BPE ranks by over-counts runs.

**Proposition 4 (rank replay).** Applying merges $1, \dots, k$ in order to a training word reproduces its
segmentation after $k$ training merges; and the greedy sequence is prefix-closed, so the first $k$ merges
of a long run are exactly the $k$-merge model.

In [ ]:
sw.test_merge_accounting()
sw.test_replay_reproduces_training_segmentation()
bpe = sw.trained()["bpe"]
print("first BPE merges:", bpe.merges[:10])
print("their pair counts:", [int(g) for g in bpe.gains[:10]])
print("aaa with (a, a):", sw.apply_merge(["a", "a", "a"], ("a", "a")))

The harness cross-checks our trainer against Hugging Face `tokenizers`: same corpus, same tie rule
(highest count, then the smaller pair of token ids), identical merge list over all 4,000 merges, and
identical encodings of every held-out word.

In [ ]:
sw.test_bpe_matches_hf_tokenizers()
print("BPE merge list == Hugging Face BpeTrainer merge list:", len(bpe.merges), "merges")

**Greedy is not optimal.** Let the *utility* of a merge sequence be the number of tokens it saves.
Finding a pair encoding of maximum utility is NP-hard — indeed APX-complete — and greedy BPE always
achieves at least $1/3$ of the optimum (Kozma and Voderholzer; Whittington, Bachmann, and Pimentel prove
NP-completeness of the decision versions). These are **cited**, not proved here. A smallest string
on which *every* greedy run loses (exhaustive search over alphabets of size $\le 3$, length $\le 9$;
`aaabaaaab` ties): `aaaabaaab` with three merges. Greedy's first merge is forced — `aa` has three replacements — and it
shatters the repeated unit `aaab` that the optimum builds as `ab`, `a·ab`, `a·aab`.

In [ ]:
s, k, ug, uo = sw.GREEDY_GAP
print(f"{s}: every greedy run saves {sw.greedy_utilities(s, k)} tokens; optimum saves {sw.optimal_pair_encoding(s, k)}")
sw.test_greedy_is_not_optimal()

## 3. WordPiece: a likelihood criterion

WordPiece (Schuster and Nakajima) merges the pair whose new unit most increases the training likelihood
under a unigram model of the current tokens. With token counts $c_t$ and $N = \sum_t c_t$, the unigram MLE
log-likelihood is $\mathcal{L} = \sum_t c_t \log c_t - N \log N$.

**Proposition 5.** If a merge makes $m$ replacements of $(a, b)$, $a \ne b$, then
$$\Delta\mathcal{L} = m\log m + (c_a - m)\log(c_a - m) + (c_b - m)\log(c_b - m) - c_a\log c_a - c_b\log c_b
- \big[(N - m)\log(N - m) - N\log N\big],$$
and $\Delta\mathcal{L} = m\big[\log\tfrac{mN}{c_a c_b} - 1\big] + R_a + R_b - R_N$ with every remainder
$R_c \in \big[0, \tfrac{m^2}{2(c - m)}\big]$. The leading term is the pair's pointwise mutual information
weighted by its count: WordPiece prefers pairs whose parts occur together far more
often than chance.

In [ ]:
sw.test_wordpiece_delta_exact()
sw.test_wordpiece_leading_term_bound()
sw.test_wordpiece_and_bpe_disagree()
wp = sw.trained()["wordpiece"]
print("BPE      :", bpe.merges[:6])
print("WordPiece:", wp.merges[:6])
print("WordPiece gains (nats):", [round(g, 1) for g in wp.gains[:6]])

The encoders differ too. BPE replays its merges; WordPiece's encoder (MaxMatch) repeatedly takes the
longest vocabulary prefix; the Unigram LM (§4) takes the most probable segmentation. Same vocabulary,
three different functions.

In [ ]:
sw.test_maxmatch_differs_from_replay_and_viterbi()
vocab = sw.merge_vocab(wp, 200 - 26)
for w in ("acquisition", "approvals"):
    print(f"{w:12} MaxMatch {sw.maxmatch_encode(w, vocab)}   replay {sw.replay_encode(w, wp.merges[:174])}")

## 4. The Unigram LM

Give each piece a probability $p(t)$, $\sum_{t \in V} p(t) = 1$, and score a segmentation by
$P(t_1, \dots, t_k) = \prod_i p(t_i)$. The marginal probability of a word sums over its lattice:
$P(x) = \sum_{\mathbf{t} \in \mathrm{Seg}_V(x)} \prod_i p(t_i)$.

**Theorem 2 (Viterbi).** With $\delta(0) = 0$ and $\delta(j) = \max_{i:\, x_{i:j} \in V} \big[\delta(i) + \log p(x_{i:j})\big]$,
$\delta(n)$ is the log-probability of the best segmentation, found in $O(nL)$ time. *(Optimal
substructure: the best segmentation of $x_{0:j}$ ending in piece $x_{i:j}$ must begin with a best
segmentation of $x_{0:i}$.)*

**Proposition 6 (forward–backward).** Replacing max by log-sum-exp gives $\log P(x)$; the posterior of a
lattice edge $(i, j)$ is $\alpha(i)\, p(x_{i:j})\, \beta(j) / P(x)$, and the edges covering any character sum to 1.

**Theorem 3 (EM).** With the vocabulary fixed, the update $p'(t) \propto \mathbb{E}[c_t]$ (expected piece
counts under the current posterior) never lowers the marginal likelihood.

Vocabulary *pruning* — dropping the pieces whose removal costs least likelihood — follows SentencePiece's
loss and is a heuristic.

In [ ]:
sw.test_viterbi_matches_brute_force()
sw.test_forward_matches_brute_force()
sw.test_edge_posteriors_cover_each_char()
sw.test_em_is_monotone()
um = sw.trained()["unigram"]
for size, m in um.items():
    print(f"|V| = {size:4d}: corpus log-likelihood {m.loglik:12.1f} nats; "
          f"nonroyaltys -> {sw.viterbi_segment('nonroyaltys', m.logp, m.max_len)[0]}")

Cross-check: SentencePiece trained on the same lines; given its piece scores, our Viterbi reproduces
its segmentation of every held-out word.

In [ ]:
sw.test_viterbi_matches_sentencepiece()
sp = sw.sentencepiece_model()
print([(w, sp.encode(w, out_type=str)) for w in ("colevys", "supplierers")])

## 5. Tokenization as compression

A unigram model is a code: piece $t$ costs $-\log_2 p(t)$ bits. The Viterbi segmentation is the shortest
encoding of $x$ under that code, and the marginal code is shorter still, but never by much.

**Proposition 7 (bits-back sandwich).** $-\log P(x) \le -\log P(\mathbf{t}^\ast) \le -\log P(x) + \log|\mathrm{Seg}_V(x)|$,
since $P(x) = \sum_{\mathbf{t}} P(\mathbf{t}) \le |\mathrm{Seg}_V(x)| \max_{\mathbf{t}} P(\mathbf{t})$.

**Two-part code (MDL).** Charge the vocabulary too: spell every piece letter by letter
($\log_2 27$ bits per letter, plus an end marker), then code the training tokens under their unigram MLE.
Along BPE's merge path the total falls, bottoms out, and rises again — subwords beat both characters
and whole words.

In [ ]:
sw.test_bits_back_sandwich()
sw.test_mdl_curve_is_exact()
sw.test_mdl_has_interior_optimum()
S = sw.finance_summary()
print(f"characters only : {S['mdl_bits_chars']:12.0f} bits")
print(f"BPE optimum     : {S['mdl_bits_min']:12.0f} bits at {S['mdl_argmin_merges']} merges (|V| = {S['mdl_vocab_at_min']})")
print(f"whole words     : {S['mdl_bits_words']:12.0f} bits")

Closing Topic 1's loop: the word vocabulary misses held-out words and over half its types are hapaxes;
a subword vocabulary encodes everything and almost none of its pieces is seen only once.

In [ ]:
sw.test_subwords_close_the_vocabulary()
for kind in ("bpe", "wordpiece", "unigram"):
    print(kind, [round(S[f"{kind}_{v}_tpw_held"], 3) for v in sw.VOCAB_SIZES], "tokens per held-out word")

## 6. Finance case study: planted morphemes

The shared corpus spells rare content words from English-like morphemes — prefixes, roots, and suffixes
(`cross·fleet`, `non·royalty·s`, `supplier·ers`). Those boundaries are a known answer. We recover them by
parsing each pseudo-word against the generator's morpheme lists (every parse turns out unique) and score
each tokenizer's internal boundaries by precision, recall, and F1, on training words and on held-out
words none of the tokenizers has seen.

Compression is not morphology: at matched vocabulary size, and even at matched tokens per word, the
Unigram LM's boundaries track the planted morphemes far better than BPE's until the vocabulary is large.

In [ ]:
sw.test_finance_rebuild_matches_corpus()
sw.test_planted_morphemes_parse_uniquely()
sw.test_unigram_tracks_planted_morphemes()
print(f"{'':10}" + "".join(f"|V|={v:<6}" for v in sw.VOCAB_SIZES))
for kind in ("bpe", "wordpiece", "unigram"):
    print(f"{kind:10}" + "".join(f"{S[f'{kind}_{v}_f1_train']:<10.3f}" for v in sw.VOCAB_SIZES), "boundary F1 (train)")
    print(f"{'':10}" + "".join(f"{S[f'{kind}_{v}_f1_oov']:<10.3f}" for v in sw.VOCAB_SIZES), "boundary F1 (held-out OOV)")
for w in ("crossfleet", "nonroyaltys", "supplierers"):
    print(w, {k: sw.segment_all(k, 400, [w])[w] for k in ("bpe", "wordpiece", "unigram")})

## 7. The lab's numbers, and the full harness

`viz_payload()` holds every number the interactive lab displays: Panel A's tokens per word and live
segmentations per merge count, and Panel B's Unigram lattices (edge posteriors, Viterbi path, and the
bits-back sandwich) per vocabulary size.

In [ ]:
P = sw.viz_payload()
for k, row in P["merges"].items():
    print(f"{k:4d} merges: BPE {row['bpe']['tpw']:.3f} / WordPiece {row['wordpiece']['tpw']:.3f} tokens per word;",
          "colevys ->", row["bpe"]["segs"]["colevys"])
for size, words in P["lattice"].items():
    d = words["nonroyaltys"]
    print(f"|V| = {size}: {d['viterbi']}  Viterbi {d['bits_viterbi']} bits, marginal {d['bits_marginal']} bits, "
          f"log2|Seg| = {d['log2_segs']}")

In [ ]:
assert sw._run_harness() == 0